In [1]:
import numpy as np

In [2]:
X_gate = np.array([[0, 1],                      # Pauli X gate
                   [1, 0]])                     # = NOT gate

Y_gate = np.array([[ 0,-1j],                    # Pauli Y gate
                   [1j,  0]])                   # = SHZHZS
  
Z_gate = np.array([[1, 0],                      # Pauli Z gate
                   [0,-1]])                     # = P(pi) = S^2
                                                # = HXH

H_gate = np.array([[1, 1],                      # Hadamard gate 
                   [1,-1]]) * np.sqrt(1/2)

S_gate = np.array([[1, 0],                      # Phase gate
                   [0,1j]])                     # = P(pi/2) = T^2
                   
T_gate = np.array([[1,                0],       # = P(pi/4)
                   [0,np.exp(np.pi/-4j)]])
                   
Tinv_gate = np.array([[1, 0],                   # = P(-pi/4) 
                      [0,np.exp(np.pi/4j)]])    # = T^-1
                      
def P_gate(phi):                                # Phase shift gate
    return np.array([[1,             0],
                     [0,np.exp(phi*1j)]])
                     
def Rx_gate(theta):                             # Y rotation gate
    return np.array([[np.cos(theta/2),-1j*np.sin(theta/2)],
                     [-1j*np.sin(theta/2),np.cos(theta/2)]])
                     
def Ry_gate(theta):                             # Y rotation gate return 
    np.array([[np.cos(theta/2),-np.sin(theta/2)],
              [np.sin(theta/2), np.cos(theta/2)]])
              
def Rz_gate(theta):                             # Z rotation gate 
    return np.array([[np.exp(-1j*theta/2),0], [0 ,np.exp(1j*theta/2)]])
                     
CNOT_gate = np.array([[1, 0, 0, 0],             # Ctled NOT gate
                      [0, 1, 0, 0],             #=XORgate
                      [0, 0, 0, 1],
                      [0, 0, 1, 0]])
                      
CZ_gate = np.array([[1, 0, 0, 0],               # Ctled Z gate
                    [0, 1, 0, 0],
                    [0, 0, 1, 0],
                    [0, 0, 0,-1]])
                    
SWAP_gate = np.array([[1, 0, 0, 0],             # Swap gate
                      [0, 0, 1, 0],
                      [0, 1, 0, 0],
                      [0, 0, 0, 1]])
                      
TOFF_gate = np.array([[1, 0, 0, 0, 0, 0, 0, 0], # Toffoli gate
                     [0, 1, 0, 0, 0, 0, 0, 0],
                     [0, 0, 1, 0, 0, 0, 0, 0],
                     [0, 0, 0, 1, 0, 0, 0, 0],
                     [0, 0, 0, 0, 1, 0, 0, 0],
                     [0, 0, 0, 0, 0, 1, 0, 0],
                     [0, 0, 0, 0, 0, 0, 0, 1],
                     [0, 0, 0, 0, 0, 0, 1, 0]])

In [14]:
def pushQubit(weights):
    global workspace
    workspace = np.reshape(workspace,(1,-1)) 
    workspace = np.kron(workspace,weights)

workspace = np.array([[1.]])       # create empty qubit stack pushQubit([1,0])
pushQubit([1,0])
print(workspace)
pushQubit([3/5,4/5])               # push a 2nd qubit print(workspace)
print(workspace)



[[1. 0.]]
[[0.6 0.8 0.  0. ]]


In [15]:
def applyGate(gate):
    global workspace
    workspace = np.reshape(workspace,(-1,gate.shape[0]))     
    np.matmul(workspace,gate.T,out=workspace)

np.matmul(X_gate,[1,0])

array([0, 1])

In [16]:
# Pauli X gate

workspace = np.array([[1.]])       # reset workspace 
pushQubit([1,0])
print("input",workspace)
applyGate(X_gate)                  # = NOT 
print("output",workspace)

input [[1. 0.]]
output [[0. 1.]]


In [17]:
# Hadamard gate

workspace = np.array([[1.]])
pushQubit([1,0])
print("input",workspace)
applyGate(H_gate)
print("output",workspace)

input [[1. 0.]]
output [[0.70710678 0.70710678]]


In [18]:
# T-gate

workspace = np.array([[1.+0j]])   
pushQubit([0,1])    # set complex workspace pushQubit([.6,.8])
print("input",workspace)
applyGate(T_gate)
print("output",workspace)

input [[0.+0.j 1.+0.j]]
output [[0.        +0.j         0.70710678+0.70710678j]]


In [19]:
# SWAP gate

workspace = np.array([[1.]])
pushQubit([1,0])                          # qubit 1
pushQubit([0.6,0.8])                      # qubit 2
print(workspace)
applyGate(SWAP_gate)
print(workspace)

[[0.6 0.8 0.  0. ]]
[[0.6 0.  0.8 0. ]]


In [20]:
def tosQubit(k):
    global workspace
    if k > 1:                                               # if non-trivial
        workspace = np.reshape(workspace,(-1,2,2**(k-1)))
        workspace = np.swapaxes(workspace,-2,-1)

workspace = np.array([[1.]])
pushQubit([1,0])
pushQubit([0.6,0.8])
print(workspace)
tosQubit(2)
print(workspace)

[[0.6 0.8 0.  0. ]]
[[[0.6 0. ]
  [0.8 0. ]]]


In [21]:
def probQubit():
    global workspace
    workspace = np.reshape(workspace,(-1,2)) 
    return np.linalg.norm(workspace,axis=0)**2
def measureQubit():
    global workspace
    prob = probQubit()
    measurement = np.random.choice(2,p=prob)         # select 0 or 1 
    workspace = (workspace[:,[measurement]]/
    np.sqrt(prob[measurement])) 
    return str(measurement)

workspace = np.array([[1. ]])
for n in range(30):
    pushQubit([0.6,0.8])
    print(measureQubit(), end="")



111011100111000011001010110010

In [22]:
workspace = np.array([[1.]]) 
for i in range(16):
    pushQubit([1,0])                      # push a zero qubit
    applyGate(H_gate)                     # set equal 0 and 1 probability
    pushQubit([1,0])                      # push a 2nd zero qubit
    applyGate(H_gate)                     # set equal 0 and 1 probability
    pushQubit([1,0])                      # push a dummy zero qubit
    applyGate(TOFF_gate)                  # compute Q3 = Q1 AND Q2
    q3 = measureQubit()                   # pop qubit 3
    q2 = measureQubit()                   # pop qubit 2
    q1 = measureQubit()                   # pop qubit 1
    print(q1+q2+q3,end=",")

111,100,000,010,010,010,010,000,100,000,000,100,100,010,000,010,

# Code improvements

In [23]:
def pushQubit(name,weights):
    global workspace
    global namestack
    if workspace.shape == (1,1):                  # if workspace empty
        namestack = []                            # then reset
    namestack.append(name)                        # push name
    weights = weights/np.linalg.norm(weights)     # normalize 
    weights = np.array(weights,dtype=workspace[0,0].dtype) 
    workspace = np.reshape(workspace,(1,-1))      # to row vector 
    workspace = np.kron(workspace,weights)

workspace = np.array([[1.]])        # create empty qubit stack 
pushQubit("Q1",[1,1])               # push a qubit 
print(np.reshape(workspace,(1,-1))) # print workspace as vector print(namestack)
pushQubit("Q2",[0,1])               # push a 2nd qubit 
print(np.reshape(workspace,(1,-1))) # print workspace as vector print(namestack)

[[0.70710678 0.70710678]]
[[0.         0.70710678 0.         0.70710678]]


In [24]:
def tosQubit(name):
    global workspace
    global namestack
    k = len(namestack)-namestack.index(name)    # qubit pos
    if k > 1:                                   # if non-trivial
        namestack.append(namestack.pop(-k))         # rotate name stack 
        workspace = np.reshape(workspace,(-1,2,2**(k-1))) 
        workspace = np.swapaxes(workspace,-2,-1)

print(np.reshape(workspace,(1,-1)))  # print workspace as vector 
print(namestack)
tosQubit("Q1")                       # swap qubits
print(np.reshape(workspace,(1,-1)))  # print workspace as vector print(namestack)
print(namestack)

[[0.         0.70710678 0.         0.70710678]]
['Q1', 'Q2']
[[0.         0.         0.70710678 0.70710678]]
['Q2', 'Q1']


In [25]:
def applyGate(gate,*names):
    global workspace
    for name in names:                   # move qubits to TOS
        tosQubit(name)
        workspace = np.reshape(workspace,(-1,gate.shape[0]))
        np.matmul(workspace,gate.T,out=workspace)

print(np.reshape(workspace,(1,-1)))       # print workspace as vector 
print(namestack)
applyGate(H_gate,"Q2")                    # H gate on qubit 2 
print(np.reshape(workspace,(1,-1)))       # turns a 0 qubit to 1 
print(namestack)                          # with 50% probability

[[0.         0.         0.70710678 0.70710678]]
['Q2', 'Q1']
[[ 0.5 -0.5  0.5 -0.5]]
['Q1', 'Q2']


In [26]:
def probQubit(name):
    global workspace
    tosQubit(name)
    workspace = np.reshape(workspace,(-1,2))
    prob = np.linalg.norm(workspace,axis=0)**2
    return prob/prob.sum()                 # make sure sum is one

def measureQubit(name): 
    global workspace 
    global namestack
    prob = probQubit(name)
    measurement = np.random.choice(2,p=prob)
    workspace = (workspace[:,[measurement]]/
                 np.sqrt(prob[measurement]))
    namestack.pop()
    return str(measurement)

workspace = np.array([[1.]])
pushQubit("Q1",[1,0])
applyGate(H_gate,"Q1")
print("Q1 probabilities:", probQubit("Q1")) # peek Q1 prob 
pushQubit("Q2",[0.6,0.8])
print("Q2 probabilities:", probQubit("Q2")) # peek Q2 prob 
print(measureQubit("Q1"), measureQubit("Q2"))

Q1 probabilities: [0.5 0.5]
Q2 probabilities: [0.36 0.64]
1 1


#### Had to be modified to move workspace = np.array([[1.+0j]]) into loop. Ran out of memory and coached without this.

In [29]:


def toffEquiv_gate(q1,q2,q3):               # define Toffoli gate
    applyGate(H_gate,q3)                    # using H, T, T*, CNOT
    applyGate(CNOT_gate,q2,q3) 
    applyGate(Tinv_gate,q3) 
    applyGate(CNOT_gate,q1,q3) 
    applyGate(T_gate,q3) 
    applyGate(CNOT_gate,q2,q3) 
    applyGate(Tinv_gate,q3) 
    applyGate(CNOT_gate,q1,q3) 
    applyGate(T_gate,q2) 
    applyGate(T_gate,q3) 
    applyGate(H_gate,q3) 
    applyGate(CNOT_gate,q1,q2) 
    applyGate(T_gate,q1) 
    applyGate(Tinv_gate,q2) 
    applyGate(CNOT_gate,q1,q2)
    
          # prep COMPLEX array
for i in range(16):          
    workspace = np.array([[1.+0j]])              # test function
    pushQubit("Q1",[1,1])
    pushQubit("Q2",[1,1])
    pushQubit("Q3",[1,0])
    toffEquiv_gate("Q1","Q2","Q3")        # compute Q3 = Q1 AND Q2
print(measureQubit("Q1")+measureQubit("Q2")+ 
      measureQubit("Q3"), end=",")

001,

In [28]:
def TOFF3_gate(q1,q2,q3,q4):
    pushQubit("temp",[1,0]) 
    applyGate(TOFF_gate,q1,q2,"temp")         
    applyGate(TOFF_gate,"temp",q3,q4) 
    applyGate(TOFF_gate,q1,q2,"temp")         # restore temp
    measureQubit("temp")                      # t is surely zero
    
workspace = np.array([[1.]])                  # test!
for i in range(20):                           # generate truth table
    pushQubit("Q1",[1,1]) 
    pushQubit("Q2",[1,1]) 
    pushQubit("Q3",[1,1]) 
    pushQubit("Q4",[1,0])                         # Q4 starts at zero so
    TOFF3_gate("Q1","Q2","Q3","Q4")               # Q4 = AND of Q1 thru Q3
    print("".join([measureQubit(q) for q in
                    ["Q1","Q2","Q3","Q4"]]), end=",")

1110,0010,0100,0010,1110,1110,1110,1110,0010,0010,0100,0110,0010,0110,0110,1000,0010,1100,1110,0000,

In [30]:
def TOFFn_gate(ctl,result): # result = result XOR AND(qubits) 
    n = len(ctl)
    if n == 0: 
        applyGate(X_gate,result)
    if n == 1: 
        applyGate(CNOT_gate,ctl[0],result)
    elif n == 2: 
        applyGate(TOFF_gate,ctl[0],ctl[1],result)
    elif n > 2: 
        k=0
        while "temp"+str(k) in namestack: 
            k=k+1
        temp = "temp"+str(k)        # generate unique name 
        pushQubit(temp,[1,0])       # push zero temp qubit 
        applyGate(TOFF_gate,ctl[0],ctl[1],temp) # apply TOFF 
        ctl.append(temp)            # add temp to controls 
        TOFFn_gate(ctl[2:],result)  # recursion 
        applyGate(TOFF_gate,ctl[0],ctl[1],temp) # uncompute temp 
        measureQubit(temp)          # pop temp
workspace = np.array([[1]],dtype=np.single)     # test!
for i in range(20):                 # generate truth table
    pushQubit("Q1",[1,1])
    pushQubit("Q2",[1,1])
    pushQubit("Q3",[1,1])
    pushQubit("Q4",[1,0])               # Q4 starts at zero, becomes 
    TOFFn_gate(["Q1","Q2","Q3"],"Q4")   # AND of Q1 thru Q3 
    print("".join([measureQubit(q) for q in
               ["Q1","Q2","Q3","Q4"]]),end=",")

1011,1010,1100,1110,1010,1000,1000,1010,1110,0100,1110,1010,0100,1100,1110,1000,0000,0000,1100,0100,

#### Same issue with the workspace = np.array([[1]],dtype=np.single) being outside the loop and crashing everything. 

In [ ]:
def applyGate(gate,*names):
    global workspace
    if list(names) != namestack[-len(names):]: # reorder stack
        for name in names: # if necessary 
            tosQubit(name)
    workspace = np.reshape(workspace,(-1,2**(len(names))))
    subworkspace = workspace[:,-gate.shape[0]:]
    np.matmul(subworkspace,gate.T,out=subworkspace)

def TOFF3_gate(q1,q2,q3,q4): 
    applyGate(X_gate,q1,q2,q3,q4)

def TOFFn_gate(ctl,result): 
    applyGate(X_gate,*ctl,result)
    

for i in range(20):
    workspace = np.array([[1]],dtype=np.single) # moved this inside loop
    pushQubit("Q1",[1,1])
    pushQubit("Q2",[1,1])
    pushQubit("Q3",[1,1])
    pushQubit("Q4",[1,0])
    
    TOFF3_gate("Q1","Q2","Q3","Q4") 
    print("".join([measureQubit(q) for q in ["Q1","Q2","Q3","Q4"]]),end="/")
    pushQubit("Q1",[1,1])
    pushQubit("Q2",[1,1]) 
    pushQubit("Q3",[1,1]) 
    pushQubit("Q4",[1,0]) 
    TOFFn_gate(["Q1","Q2","Q3"],"Q4") 
    print("".join([measureQubit(q) for q in ["Q1","Q2","Q3","Q4"]]),end=",")

1000/0000,1100/1000,1111/0100,1000/1010,0100/1000,0000/0010,1000/0010,0010/1010,0100/1000,0110/1111,1100/1100,1010/1100,0110/1010,0100/1100,1000/1100,1111/0100,1111/1111,0010/1000,0110/0010,1010/1000,

# Grover's Search

In [35]:
def zero_booleanOracle(qubits,result): # all qubits zero? 
    # if all qubits==0 return 1 else return 0
    for qubit in qubits:             # negate all inputs
        applyGate(X_gate,qubit)
    TOFFn_gate(qubits,result)        # compute AND
    for qubit in qubits:             # restore inputs
        applyGate(X_gate,qubit)

def zero_phaseOracle(qubits):            # all qubits zero? 
    # if all qubits==0 return -weight else return weight
    for qubit in qubits:                 # negate all inputs
        applyGate(X_gate,qubit)
    applyGate(Z_gate,*namestack)         # controlled Z gate
    for qubit in qubits:                 # restore inputs
        applyGate(X_gate,qubit)

def sample_phaseOracle(qubits):          # sample function 
        # if all f(x)==1 return -weight else return weight
    applyGate(X_gate,qubits[1])          # negate qubit 1
    applyGate(Z_gate,*namestack)         # controlled Z gate
    applyGate(X_gate,qubits[1])          # restore qubit 1

def groverSearch(n, printProb=True):
    optimalTurns = int(np.pi/4*np.sqrt(2**n)-1/2)   # iterations 
    qubits = list(range(n))                         # generate qubit names
    for qubit in qubits:                            # initialize qubits
        pushQubit(qubit,[1,1])
    for k in range(optimalTurns):                   # Grover iterations:
        sample_phaseOracle(qubits)                  # apply phase oracle
        for qubit in qubits:                        # H-gate all qubits
            applyGate(H_gate,qubit) 
        zero_phaseOracle(qubits)                    # apply 0 phase oracle
        for qubit in qubits:                        # H-gate all qubits
            applyGate(H_gate,qubit) 
        if printProb:                               # peek probabilities
            print(probQubit(qubits[0]))             # to show convergence
    for qubit in reversed(qubits):                  # print result 
        print(measureQubit(qubit),end="")

In [36]:
workspace = np.array([[1.]])              # initialize workspace 
groverSearch(6)                           # (only reals used here)

[0.43945312 0.56054687]
[0.33325958 0.66674042]
[0.20755294 0.79244706]
[0.09326882 0.90673118]
[0.01853182 0.98146818]
111101